# Qwen3 top-k teacher logits on TPU

Produces the distillation artefact described in `docs/tasks/tpu-topk-teacher.md`:
one record per **teacher** token — top-k ids, top-k values, and the token's byte
offset — so a 32k student can align by byte span later.

**Refuses to run without a TPU.** A CPU run is not a substitute and is not
recorded as a measurement.

Order of work: smoke (0.6B, 1M tokens, k=8) → real (1.7B, 22.7M tokens, k=32).


In [ ]:
# Kaggle's TPU image usually ships torch_xla; install only if the import fails.
try:
    import torch_xla  # noqa: F401
    print('torch_xla present')
except ImportError:
    !pip install -q torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html 2>&1 | tail -3
    import torch_xla  # noqa: F401


In [ ]:
import torch
import torch_xla
import torch_xla.core.xla_model as xm

# Ask XLA, not /dev: a Kaggle TPU VM exposes no /dev/accel* and sets no TPU_NAME.
dev = xm.xla_device()
hw = xm.xla_device_hw(dev)
print('xla device:', dev, 'hardware:', hw)
assert hw == 'TPU', f'expected a TPU, got {hw}. A CPU session is not an acceptable substitute.'


In [ ]:
# SMOKE=1 runs the 0.6B teacher on a small slice first. Do not skip it.
SMOKE = os.environ.get('SMOKE', '1') == '1'
TEACHER = 'Qwen/Qwen3-0.6B' if SMOKE else 'Qwen/Qwen3-1.7B'
K = 8 if SMOKE else 32
MAX_TOKENS = 1_000_000 if SMOKE else 22_700_000

from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(TEACHER)
model = AutoModelForCausalLM.from_pretrained(TEACHER, torch_dtype=torch.bfloat16)
model = model.to(xm.xla_device()).eval()
print('teacher', TEACHER, 'vocab', len(tok), 'k', K, 'budget', MAX_TOKENS)


In [ ]:
# Train shards only: the val and heldout splits must never reach a teacher dump
# that the student trains on. The u16 ids come from a lossless byte tokenizer, so
# decoding recovers the exact text; that is asserted before any token is trusted.
import glob, struct
from pathlib import Path

def find(pattern):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))
    return hits

TRAIN = find('train-*.bin') or find('*.bin')
assert TRAIN, 'attach sectionme/ferrisres-corpus-release-slice; no train shard under /kaggle/input'
print('train shards:', len(TRAIN))
for p in TRAIN:
    print('  ', p, Path(p).stat().st_size)

def read_u16(path):
    raw = Path(path).read_bytes()
    if len(raw) % 2:
        raise SystemExit(f'{path}: odd byte count, not u16 ids')
    return list(struct.unpack('<%dH' % (len(raw) // 2), raw))

def byte_decode(ids):
    return bytes(i & 0xFF for i in ids).decode('utf-8', errors='strict')

probe = read_u16(TRAIN[0])[:100_000]
text_probe = byte_decode(probe)
assert [b for b in text_probe.encode('utf-8')] == probe[:len(text_probe.encode('utf-8'))], \
    'byte tokenizer round trip failed; stop, the shards are not what we think'
print('round trip ok on', len(text_probe.encode('utf-8')), 'bytes')


In [ ]:
import time, json
import numpy as np

# Fixed window = fixed XLA shape = one compile. A ragged tail is padded and its
# true length recorded, so the student side never reads padding as tokens.
WINDOW = 1024
out_dir = Path('/kaggle/working/topk')
out_dir.mkdir(exist_ok=True)
for old in out_dir.glob('part-*.bin'):
    old.unlink()

dev = xm.xla_device()
pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id

def run_window(chunk):
    ids = torch.tensor([chunk], dtype=torch.long, device=dev)
    with torch.no_grad():
        out = model(input_ids=ids)
    xm.mark_step()
    return out.logits[0].float().cpu()

total_tokens = 0
part = 0
t_start = time.perf_counter()
t_compile = None
parts = []

for shard in TRAIN:
    if total_tokens >= MAX_TOKENS:
        break
    ids = read_u16(shard)
    text = byte_decode(ids)
    enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)
    tids, offs = enc['input_ids'], enc['offset_mapping']
    print(shard, 'ids', len(ids), 'teacher tokens', len(tids))
    for i in range(0, len(tids), WINDOW):
        if total_tokens >= MAX_TOKENS:
            break
        chunk = tids[i:i + WINDOW]
        span = offs[i:i + WINDOW]
        n = len(chunk)
        padded = chunk + [pad_id] * (WINDOW - n)
        t0 = time.perf_counter()
        logits = run_window(padded)
        if t_compile is None:
            t_compile = time.perf_counter() - t0
        vals, idx = logits[:n].topk(K, dim=-1)
        assert torch.all(vals[:, :-1] >= vals[:, 1:]), 'topk not sorted'
        rec = np.zeros(n, dtype=[('idx', '<i4', (K,)), ('val', '<f2', (K,)), ('off', '<i4')])
        rec['idx'] = idx.numpy().astype('<i4')
        rec['val'] = vals.numpy().astype('<f2')
        rec['off'] = np.array([o[0] for o in span], dtype='<i4')
        assert (np.diff(rec['off']) >= 0).all(), 'offsets must be non-decreasing'
        name = f'part-{part:05d}.bin'
        rec.tofile(out_dir / name)
        parts.append({'part': name, 'tokens': int(n), 'shard': Path(shard).name})
        total_tokens += n
        part += 1
        if part % 50 == 0:
            print(f'{total_tokens} tokens, {part} parts, {time.perf_counter() - t_start:.0f}s')

elapsed = time.perf_counter() - t_start
manifest = {
    'teacher': TEACHER, 'k': K, 'vocab': int(len(tok)), 'window': WINDOW,
    'dtype': 'idx:int32,val:bf16,off:int32',
    'tokens': total_tokens, 'parts': parts,
    'bytes': sum((out_dir / p['part']).stat().st_size for p in parts),
    'compile_seconds': round(t_compile or 0.0, 2),
    'run_seconds': round(elapsed, 2),
    'tokens_per_s': round(total_tokens / elapsed, 1) if elapsed else 0.0,
    'smoke': SMOKE,
}
(out_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps({k: v for k, v in manifest.items() if k != 'parts'}, indent=2))
